In [22]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-mlp-wine-q"

## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [23]:
train_loader, test_loader = data_load_and_prep("wine", test_size=0.3, random_state=42, batch_size='full')
NUM_EPOCHS = 30
HIDDEN_SIZES = [6, 4]

# Classical optimization for benchmarking

In [4]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cpu" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [5]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/03 13:08:22 INFO mlflow.tracking.fluent: Experiment with name 'optimizer-comparison-mlp-wine-q' does not exist. Creating a new experiment.


2026/05/03 13:08:22 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.0693 | train_acc=0.532 | test_loss=1.0635 | test_acc=0.519 | 
Epoch 005 | train_loss=0.9555 | train_acc=0.677 | test_loss=0.9470 | test_acc=0.722 | 
Epoch 010 | train_loss=0.8345 | train_acc=0.710 | test_loss=0.8242 | test_acc=0.778 | 
Epoch 015 | train_loss=0.7005 | train_acc=0.839 | test_loss=0.6921 | test_acc=0.907 | 
Epoch 020 | train_loss=0.5625 | train_acc=0.935 | test_loss=0.5650 | test_acc=0.926 | 
Epoch 025 | train_loss=0.4327 | train_acc=0.976 | test_loss=0.4545 | test_acc=0.926 | 
Epoch 029 | train_loss=0.3433 | train_acc=1.000 | test_loss=0.3807 | test_acc=0.963 | 


{'run_id': '7c0f51b142754b95923892a450f7b973',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.01781888800132947,
 'raw_optimization_time_sec': 0.1888828750011271,
 'sum_epoch_optimization_time_sec': 0.01781888800132947,
 'train_evaluation_time_sec': 0.022023424000508385,
 'test_evaluation_time_sec': 0.01280762099850108,
 'evaluation_time_sec': 0.034831044999009464,
 'training_time_sec': 0.3298409180001727,
 'final_train_loss': 0.3433355689048767,
 'final_test_loss': 0.3807152509689331,
 'final_train_metric': 1.0,
 'final_test_metric': 0.9629629629629629,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Adam optimizaer on cuda

In [6]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

In [7]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer-cuda"
)

2026/05/03 13:08:35 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.0403 | train_acc=0.331 | test_loss=1.0303 | test_acc=0.333 | 
Epoch 005 | train_loss=0.8487 | train_acc=0.710 | test_loss=0.8311 | test_acc=0.685 | 
Epoch 010 | train_loss=0.6896 | train_acc=0.831 | test_loss=0.6733 | test_acc=0.870 | 
Epoch 015 | train_loss=0.5644 | train_acc=0.887 | test_loss=0.5539 | test_acc=0.889 | 
Epoch 020 | train_loss=0.4649 | train_acc=0.927 | test_loss=0.4599 | test_acc=0.926 | 
Epoch 025 | train_loss=0.3813 | train_acc=0.944 | test_loss=0.3838 | test_acc=0.981 | 
Epoch 029 | train_loss=0.3227 | train_acc=0.944 | test_loss=0.3329 | test_acc=0.981 | 


{'run_id': '428fb21d121e4fa1987c5dcbe139083f',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.028976985000554123,
 'raw_optimization_time_sec': 0.19803344900083175,
 'sum_epoch_optimization_time_sec': 0.028976985000554123,
 'train_evaluation_time_sec': 0.04960109600006035,
 'test_evaluation_time_sec': 0.016300375999890093,
 'evaluation_time_sec': 0.06590147199995045,
 'training_time_sec': 0.34875431299997217,
 'final_train_loss': 0.3227112591266632,
 'final_test_loss': 0.3329440653324127,
 'final_train_metric': 0.9435483870967742,
 'final_test_metric': 0.9814814814814815,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using LBFGS-style second-order optimizer

In [8]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cpu" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [9]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

2026/05/03 13:08:48 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.2848 | train_acc=0.266 | test_loss=1.2895 | test_acc=0.278 | 
Epoch 005 | train_loss=0.9750 | train_acc=0.468 | test_loss=0.9979 | test_acc=0.407 | 
Epoch 010 | train_loss=0.8904 | train_acc=0.573 | test_loss=0.9184 | test_acc=0.537 | 
Epoch 015 | train_loss=0.7775 | train_acc=0.782 | test_loss=0.8114 | test_acc=0.741 | 
Epoch 020 | train_loss=0.6726 | train_acc=0.790 | test_loss=0.7114 | test_acc=0.778 | 
Epoch 025 | train_loss=0.5557 | train_acc=0.863 | test_loss=0.5999 | test_acc=0.852 | 
Epoch 029 | train_loss=0.4687 | train_acc=0.944 | test_loss=0.5168 | test_acc=0.926 | 


{'run_id': '908def5c309749c7b1ce0affd9375009',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.01875997100114546,
 'raw_optimization_time_sec': 0.02177280300111306,
 'sum_epoch_optimization_time_sec': 0.01875997100114546,
 'train_evaluation_time_sec': 0.018219294998743862,
 'test_evaluation_time_sec': 0.01106401400102186,
 'evaluation_time_sec': 0.029283308999765723,
 'training_time_sec': 0.14101306100019428,
 'final_train_loss': 0.46870285272598267,
 'final_test_loss': 0.5167831182479858,
 'final_train_metric': 0.9435483870967742,
 'final_test_metric': 0.9259259259259259,
 'optimizer': 'LBFGS',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Newton-Rapson method

In [10]:
loss_fn = nn.CrossEntropyLoss() 
newton_model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cpu"
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                                   lr=1, 
                                   max_iter=1,
                                   damping=1e-4,
                                   )

In [11]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.1035 | train_acc=0.363 | test_loss=1.1090 | test_acc=0.259 | 
Epoch 005 | train_loss=4142.7427 | train_acc=0.290 | test_loss=5163.6187 | test_acc=0.241 | 
Epoch 010 | train_loss=1578.4557 | train_acc=0.452 | test_loss=1902.1969 | test_acc=0.296 | 
Epoch 015 | train_loss=3279.3308 | train_acc=0.363 | test_loss=3013.9353 | test_acc=0.370 | 
Epoch 020 | train_loss=3066.6230 | train_acc=0.653 | test_loss=4341.5303 | test_acc=0.593 | 
Epoch 025 | train_loss=3046.9998 | train_acc=0.411 | test_loss=2893.6292 | test_acc=0.352 | 


2026/05/03 13:08:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=2935.0303 | train_acc=0.573 | test_loss=4335.9731 | test_acc=0.556 | 


{'run_id': '4ffa9ea88fdd4d5d8ee510e973839468',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 1.6247152470009496,
 'raw_optimization_time_sec': 1.6902029610009777,
 'sum_epoch_optimization_time_sec': 1.6247152470009496,
 'train_evaluation_time_sec': 0.01850839100052326,
 'test_evaluation_time_sec': 0.010855918000288511,
 'evaluation_time_sec': 0.029364309000811772,
 'training_time_sec': 1.8133559689999856,
 'final_train_loss': 2935.0302734375,
 'final_test_loss': 4335.97314453125,
 'final_train_metric': 0.5725806451612904,
 'final_test_metric': 0.5555555555555556,
 'optimizer': 'NewtonOptimizer',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Quadratic Annealer (cpu + momentum)

In [24]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cpu" 

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=36,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [25]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-optimizer"
)

Epoch 000 | train_loss=1.0348 | train_acc=0.331 | test_loss=1.0555 | test_acc=0.333 | 
Epoch 005 | train_loss=0.0966 | train_acc=0.992 | test_loss=0.1388 | test_acc=0.981 | 
Epoch 010 | train_loss=0.0196 | train_acc=1.000 | test_loss=0.1428 | test_acc=0.963 | 


2026/05/03 21:58:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': '0c2883535e6647b0a4110fc1f42c652d',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.9894176989910193,
 'raw_optimization_time_sec': 1.0465808179797023,
 'sum_epoch_optimization_time_sec': 0.9894176989910193,
 'train_evaluation_time_sec': 0.012147601002652664,
 'test_evaluation_time_sec': 0.006421632009733003,
 'evaluation_time_sec': 0.018569233012385666,
 'training_time_sec': 1.1605296870038728,
 'final_train_loss': 0.019586995244026184,
 'final_test_loss': 0.14275358617305756,
 'final_train_metric': 1.0,
 'final_test_metric': 0.9629629629629629,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 15,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_6_consecutive_epochs',
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 6}

## Optimization using Quadratic Annealer (D-Wave + momentum)

In [20]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(13, HIDDEN_SIZES, 3)
classical_device = "cpu" 

sampler = build_sampler(mode="dwave")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=36,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [21]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-dwave-optimizer"
)

Epoch 000 | train_loss=1.0654 | train_acc=0.411 | test_loss=1.0813 | test_acc=0.259 | 
Epoch 005 | train_loss=0.2128 | train_acc=0.968 | test_loss=0.2539 | test_acc=0.963 | 
Epoch 010 | train_loss=0.0023 | train_acc=1.000 | test_loss=0.0598 | test_acc=0.981 | 
Epoch 015 | train_loss=0.0012 | train_acc=1.000 | test_loss=0.0643 | test_acc=0.981 | 


2026/05/03 13:14:57 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': '50857e178ba94decbd94800411420f30',
 'experiment_id': '671235730259710864',
 'experiment_name': 'optimizer-comparison-mlp-wine-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.6317611150002449,
 'raw_optimization_time_sec': 2.8457570150007996,
 'sum_epoch_optimization_time_sec': 0.6317611150002449,
 'train_evaluation_time_sec': 0.015375636000953818,
 'test_evaluation_time_sec': 0.007635599000877846,
 'evaluation_time_sec': 0.023011235001831665,
 'training_time_sec': 21.86442288399985,
 'final_train_loss': 0.0012111614923924208,
 'final_test_loss': 0.06430421024560928,
 'final_train_metric': 1.0,
 'final_test_metric': 0.9814814814814815,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 18,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_6_consecutive_epochs',
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 6}